In [1]:
from src.config import cfg
import re
import os
import string
import unicodedata
from collections import Counter
from datasets import load_dataset

In [3]:
ds = load_dataset("parquet", data_files=cfg.dataset.path, split="train")

In [4]:
def clean_text(text):
    text = unicodedata.normalize("NFC", text)

    text = text.lower().replace("i\u0307", "i")

    text = re.sub(r"[\'\"`’‘“”]", "", text)

    # sayı-sayı arasındaki tire → boşluk
    text = re.sub(r"(?<=\d)-(?=\d)", " ", text)

    # harf-harf olmayan tireler → boşluk
    text = re.sub(r"(?<!\w)[-–—]|[-–—](?!\w)", " ", text)

    text = re.sub(r"[^\w\s\-çğıöşü]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [5]:
def process_data(ds=ds):
    path = cfg.dataset.path

    def process_batch(batch):
        cleaned = [clean_text(text) for text in batch["text"]]
        tokens = [text.split(" ") for text in cleaned]
        return {"text": cleaned, "tokens": tokens}

    num_proc = os.cpu_count()
    ds = ds.map(process_batch, batched=True, batch_size=1000, num_proc=num_proc)

    texts = ds["text"]
    tokens = ds["tokens"]

    return texts, tokens
data,tokens=process_data()

Map (num_proc=8):   0%|          | 0/1005338 [00:00<?, ? examples/s]

In [6]:
idx = 2
step = 0
ad=50
b,e = step*ad,step*ad+ad
print(tokens[idx][b:e])

['mehmed', 'mustafa', 'subhi', 'kısaca', 'mustafa', 'suphi', 'veya', 'bazı', 'kaynaklarda', 'kullanıldığı', 'haliyle', 'osmanlıca', 'yazıma', 'göre', 'mustafa', 'subhi', '4', 'ağustos', '1882', 'veya', '4', 'mayıs', '1883', '28', 'ocak', '1921', 'türk', 'komünist', 've', 'türkiye', 'komünist', 'partisinin', 'ilk', 'merkez', 'komitesi', 'başkanı', 'yaşamı', 'erken', 'dönemler', 'ailesi', 'aslen', 'samsunlu', 'bir', 'aileden', 'gelmektedir', '2', 'şubat', '1895', 'tarihli', 'sicill-i']


In [7]:
counter = Counter()
for token in tokens:
    counter.update(token)

In [ ]:
counter